# The locus of generic eigenvector varieties

For generic $\mathcal{H}$, the eigenvector variety $\mathcal{E}(\mathcal{H})$ is a degree-$n$ hypersurface in $\mathbb{P}^{n-1}$ with the $n\times n$ linear determinantal representation
$$f(x_1,\dots,x_n) \;=\; \det\bigl[\,H_1\, x \mid \cdots \mid H_d\, x \mid x\,\bigr], \qquad H_1,\dots,H_d\in\mathbb{C}^{n\times n},\quad d=n-1.$$

Let $\mathcal{L}\subset\mathbb{P}^{\binom{2n-1}{n}-1}$ denote the *locus of generic eigenvector varieties*. It is the Zariski closure of the set of such $f$ obtained from generic tuples $(H_1,\dots,H_d)$.  By Corollary 2.5 in our paper, for $n\ge 4$ we get
$$\dim\mathcal{L} \;=\; n^3 - 2n^2 + 1.$$

This notebook verifies $\dim\mathcal{L}$ numerically and computes $\deg\mathcal{L}$ numerically via monodromy in `HomotopyContinuation.jl`.

The smallest case $(d,n)=(3,4)$ is the default below.  This is the locus $\mathcal{F}_1$ defined in [LLHV].  It is an irreducible *hypersurface* in $\mathbb{P}^{34}$.  The defining polynomial has degree $320{,}112$. This number was obtained via intersection theory in [LLHV, Theorem 2].  We recover this number numerically by intersecting $\mathcal{L}$ with a random line in $\mathbb{P}^{34}$ and counting parameter fibers using `monodromy_solve`.

**Pipeline:**
1. Build the parametrization $(H_1,\dots,H_d)\mapsto P=(\text{coeffs of }f)$ of $\mathcal{L}$.
2. Obtain a finite-to-one parametrization $Z$ of $\mathcal{L}$.  Verify numerically that the Jacobian rank does not drop.
3. Cut $\mathcal{L}$ with a generic affine-linear subspace of complementary dimension to $\mathcal{L}$, given by $\{c\cdot[1;Z]=0\}$, and find one starting fiber.
4. Run `monodromy_solve` to count all preimages of the cut.

**Reference.** M. Leal, C. Lozano Huerta, M. Vite, *The Noether–Lefschetz locus of surfaces in $\mathbb{P}^3$ formed by determinantal surfaces*, arXiv:[2303.09028](https://arxiv.org/abs/2303.09028).

In [ ]:
using HomotopyContinuation
using LinearAlgebra

# ─── problem size ──────────────────────────────────────────
n = 4         # number of variables x_1, ..., x_n  (assume n ≥ 4)
d = n - 1     # number of matrices H_1, ..., H_d  (forced by square determinant)
@assert n >= 4 "This notebook assumes n ≥ 4."

@var x[1:n]
@var H[1:d, 1:n, 1:n]

coeffs = vec(H);   # d·n² entries of (H_1, ..., H_d)

## Building $f$ and its coefficient vector $P$

Form $f$ and take $P\in\mathbb{C}^{\binom{2n-1}{n}}$, the coefficient vector of $f$ in the monomial basis of degree-$n$ forms in $n$ variables.  It parametrizes the locus $\mathcal{L}$.  The numerical rank of the Jacobian of $P$ is the affine dimension of $\mathcal{L}$, equal to $\dim\mathcal{L}+1 = n^3 - 2n^2 + 2$ (so $34$ at the default $n=4$).

In [ ]:
# Columns of the determinantal matrix:  [H_1 x | ... | H_d x | x]
columns = [H[i, :, :] * x for i in 1:d]
push!(columns, x)
mat = hcat(columns...)

f = expand(det(mat))
P = coefficients(f, x)

# The affine dimension of L equals the numerical rank of the Jacobian of P
J = differentiate(P, coeffs)
println("affine dim of L = ",
        rank(evaluate.(J, coeffs => randn(length(coeffs)))))

## Finite-to-one parametrization

Set one row of each of $H_1,\dots,H_d$ to zero and a further $d-1$ entries to $1$.  Let $Z$ be the resulting coefficient vector.  This yields a finite-to-one parametrization of $\mathcal{L}$ as long as the Jacobian rank does not drop.  We verify it numerically.

In [ ]:
# Row n+1-i of H_i set to 0; one entry of H_i set to 1 (for i = 1, ..., d-1)
elim = vcat([H[i, n+1-i, :] for i in 1:d]...)
os   = [H[i, n-i, 1] for i in 1:d-1]

Z    = evaluate.(P, [elim; os] => [zeros(length(elim)); ones(length(os))])
Jz   = differentiate(Z, coeffs)
dimL = rank(evaluate.(Jz, coeffs => randn(length(coeffs))))

println("affine dim of L (after chart) = ", dimL)

## Start fiber

A generic affine-linear subspace of complementary dimension to $\mathcal{L}$ pulls back to the system $c\cdot[1;Z]=0$, linear in $c$.  We find a start pair by sampling a $\text{start\_sol}$ and solving for $\text{start\_params}$ by linear algebra.

In [ ]:
vars_Z = variables(System(Z))      # remaining entries of (H_1, ..., H_d) after the chart

@var c[1:dimL, 1:length(Z) + 1]
c_vec = vec(c)

# Parametric affine-linear cut:  F(x; c) = c · [1; Z(x)]
F = System(c * [1; Z], parameters = c_vec)

# Sample a start point in vars_Z-space and require the cut to vanish there.
# The resulting equations are linear in c_vec, so we solve them by linear algebra.
start_sol = rand(ComplexF64, length(vars_Z))
param_eqs = evaluate(F, start_sol)              # vector of polys in c_vec

M = convert(Matrix{ComplexF64},
            [differentiate(p, q) for p in param_eqs, q in c_vec])
b = evaluate.(param_eqs, c_vec => zeros(length(c_vec)))

# Particular solution of  M·c = -b  perturbed within the null space
N = nullspace(M)
start_params = M \ (-b) + N * rand(ComplexF64, size(N, 2))

println("residual at start fiber: ",
        norm(evaluate.(param_eqs, c_vec => start_params)))

## Monodromy

Starting from the start fiber, `monodromy_solve` finds all parameter solutions of the affine cut by tracking loops in $c$-space.  The function $\mathrm{dist}(u,v)=\|G(u)-G(v)\|_\infty$ identifies parameter solutions whose images on the cut agree.  The count of distinct images is $\deg\mathcal{L}$.  (This computation is slow.  We recommend running it on many threads (>100) and on a server.)

In [ ]:
G    = System(Z, variables = vars_Z)
dist = (u, v) -> norm(G(u) - G(v), Inf)

result = monodromy_solve(F, start_sol, start_params;
                         distance = dist,
                         threading = true,
                         catch_interrupt = true)

## Verify distinct images

`monodromy_solve`'s `dist` does not always merge solutions with the same image.  Recompare by rounding $G(u)$ entrywise to tolerance $10^{-5}$ and counting unique values.  The result is $\deg\mathcal{L}$.

In [ ]:
# Two solutions agree on the cut iff their images G(u) coincide.
# Compare entry-wise with tolerance 1e-5.
sols = solutions(result)
key(s) = Tuple((round(real(z), digits=5), round(imag(z), digits=5)) for z in G(s))
unique_sols = unique(key, sols)

println("raw solutions: ", length(sols))
println("deg L       = ", length(unique_sols))